# HLA Inference from TCRβ Repertoires — THNet on Rosati 2022 Cohort

**Method**: THNet v1.0.3 (Pan et al. 2025) — HLA typing from TCRβ bulk sequencing using sequence-embedding-based classifiers trained on 4,144 HLA-genotyped individuals.

**Dataset**: Rosati et al. 2022 (PRJEB50045) — 267 patients with bulk TCRαβ sequencing across four clinical groups: Healthy controls (n=99), Crohn's Disease (CD, n=114), Ulcerative Colitis (UC, n=40), Colorectal Cancer (CRC, n=14 after QC).

**Context**: THNet is applied as an independent second method alongside HLAGuessr (Ruiz Ortega et al. 2025) to cross-validate predicted HLA genotypes. THNet's training set (Pan et al. 2025) is **entirely independent of and disjoint from this cohort** — unlike HLAGuessr, which used Rosati's cohort as part of its own training data (see `rosati_hlaguessr.ipynb`). THNet's predictions here are therefore genuinely blind, making this the more informative result when later validated against ground-truth HLA (see `rosati_qc_hla_validation.ipynb`).

**Key methodological differences vs HLAGuessr**:
- THNet concatenates CDR3β + V-gene as sequence features (embedding-based approach)
- HLAGuessr performs exact matching of HLA-associated public clonotypes (occurrence-based)
- THNet covers 208 alleles; HLAGuessr covers 94 — all 94 HLAGuessr alleles are a strict subset of THNet
- THNet uses TCRβ only; HLAGuessr uses both α and β chains

**Reference**: Pan et al. 2025, *Quantitative and large-scale investigation of human TCR-HLA cross-reactivity*, GitHub: github.com/Mia-yao/THNet

## 0. Setup & paths

In [ ]:
# pip install THNet

import pandas as pd
import numpy as np
import os
import time
import warnings
from THNet.HLA_inference import HLA_inference
from THNet.HLA_inference.model_prediction import Model_prediction

warnings.filterwarnings('ignore')

BASE          = '/home/immunologylab/bioinformatics/'
PROCESSED_CSV = BASE + 'hla-tcr-project/data/processed/rosati_repertoires_processed.csv'
RESULTS_DIR   = BASE + 'hla-tcr-project/results/'
THNET_INPUT_CLEAN = RESULTS_DIR + 'thnet_rosati_input_clean.csv'
THNET_OUTPUT  = RESULTS_DIR + 'thnet_rosati_output/'
HLAG_PREDS    = RESULTS_DIR + 'hlaguessr_rosati_predictions.csv'
CROSSVAL_CSV  = RESULTS_DIR + 'hlaguessr_46B_crossval.csv'

os.makedirs(THNET_OUTPUT, exist_ok=True)

def get_group(pid):
    if pid.startswith('CRC_'):      return 'CRC'
    elif pid.startswith('CD_'):     return 'CD'
    elif pid.startswith('UC_'):     return 'UC'
    elif pid.startswith('Healthy'): return 'Healthy'
    else: return 'Other'

## 1. Model inspection & allele coverage

THNet expects a CSV with three columns: `cdr3` (CDR3β amino acid sequence), `v_gene` (IMGT V-gene, full notation e.g. `TRBV6-5`), and `sample` (patient identifier). Unlike HLAGuessr, no family-level V-gene normalisation is required.

The model covers **208 HLA alleles** across all classical loci. All 94 alleles modelled by HLAGuessr are a strict subset of THNet's allele space, enabling direct comparison on shared alleles.

In [ ]:
THNET_PKG = BASE + 'soft/miniconda3/envs/bioinf/lib/python3.12/site-packages/THNet/'

# Inspect expected input format
example = pd.read_csv(THNET_PKG + 'HLA_inference/example/input_example.csv')
print(f'Example input format:')
print(f'  Shape: {example.shape} | Columns: {list(example.columns)}')
print(example.head(5).to_string())

# Initialise models
mp = Model_prediction()
hi = HLA_inference()

print(f'\nTHNet alleles in model: {len(hi.hla_list)}')
print(f'THNet V-genes in model: {len(mp.v_gene_list)}')

# Allele overlap with HLAGuessr
thnet_alleles     = {a.replace('HLA-','') for a in hi.hla_list}
hlaguessr_alleles = set(pd.read_csv(CROSSVAL_CSV)['HLA'].unique())
shared     = thnet_alleles & hlaguessr_alleles
thnet_only = thnet_alleles - hlaguessr_alleles
hlag_only  = hlaguessr_alleles - thnet_alleles

print(f'\nAllele overlap (HLAGuessr ∩ THNet): {len(shared)}')
print(f'THNet only:                         {len(thnet_only)}')
print(f'HLAGuessr only (not in THNet):      {len(hlag_only)}')
print(f'All HLAGuessr alleles covered by THNet: {"yes" if len(hlag_only)==0 else "no"}')

## 2. Prepare input — TCRβ only

THNet operates exclusively on TCRβ chains. The processed Rosati repertoire CSV contains both TRA and TRB; only TRB clones are used. V-genes are passed as-is since THNet's vocabulary covers all V-genes present in the Rosati dataset.

**V-gene compatibility check**: `TRBV7-5` (310 clones, 0.003% of total, 136 patients) is absent from THNet's training vocabulary and is removed. The impact on downstream results is negligible.

In [ ]:
rosati = pd.read_csv(PROCESSED_CSV)

# TCRβ only — rename to THNet column format
beta = rosati[rosati['chain']=='TRB'][['cdr3aa','v_gene','patient_id']]\
       .rename(columns={'cdr3aa':'cdr3','patient_id':'sample'})\
       .dropna(subset=['cdr3','v_gene'])

print(f'TCRβ clones total:   {len(beta):,}')
print(f'Patients:            {beta["sample"].nunique()}')

# V-gene compatibility
thnet_vgenes   = set(mp.v_gene_list)
invalid_vgenes = set(beta['v_gene'].unique()) - thnet_vgenes
print(f'\nInvalid V-genes: {sorted(invalid_vgenes)}')
for vg in invalid_vgenes:
    n = (beta['v_gene']==vg).sum()
    print(f'  {vg}: {n:,} clones ({n/len(beta)*100:.4f}%) — removed')

beta_clean = beta[beta['v_gene'].isin(thnet_vgenes)].copy()
beta_clean.to_csv(THNET_INPUT_CLEAN, index=False)

print(f'\nClones after filter: {len(beta_clean):,}')
print(f'Saved: {THNET_INPUT_CLEAN}')

## 3. Sanity check — single patient (CD_1)

Before running the full cohort, the API is validated on patient CD_1 using its complete TCRβ repertoire. A 500-clone subset yields only 1 carrier call, while the full ~12k-clone repertoire yields 7 — confirming that **sufficient repertoire depth is essential for reliable inference**.

**DPA1*01:03 artefact identified**: this allele is predicted True in 100% of patients (minimum score 0.754). It is a model artefact caused by class imbalance during training (~95% DPA1*01:03 frequency in European training data). This allele is **excluded from all downstream analyses**.

In [ ]:
beta_clean  = pd.read_csv(THNET_INPUT_CLEAN)
all_patients = sorted(beta_clean['sample'].unique())

cd1      = beta_clean[beta_clean['sample']=='CD_1']
pred_raw = mp.Get_prediction(cd1)
scores   = pred_raw['CD_1']
calls    = hi.convert_pred(list(scores.values()))

df_cd1 = pd.DataFrame({
    'HLA':               [a.replace('HLA-','') for a in scores.keys()],
    'score':             list(scores.values()),
    'predicted_carrier': calls.astype(bool)
})

print(f'CD_1 — {len(cd1):,} clones | {df_cd1["predicted_carrier"].sum()} alleles predicted True')
print()
print('Predicted carriers:')
print(df_cd1[df_cd1['predicted_carrier']==True]
      [['HLA','score']].sort_values('score', ascending=False).to_string(index=False))
print()
print(f'DPA1*01:03 score: {scores["HLA-DPA1*01:03"]:.3f} — excluded from all downstream analyses')

## 4. Full cohort inference — 267 patients

THNet is applied to all 267 patients. Predictions are cached per patient to allow resumption if interrupted. Runtime: ~21 minutes on a 28-core server (single-threaded, Python API).

In [ ]:
results = []
t0      = time.time()

for i, pat in enumerate(all_patients):
    out_file = THNET_OUTPUT + f'{pat}_thnet.csv'

    if os.path.exists(out_file):  # resume
        results.append(pd.read_csv(out_file))
        continue

    pat_df = beta_clean[beta_clean['sample']==pat]
    if len(pat_df) == 0:
        continue

    try:
        pred_raw = mp.Get_prediction(pat_df)
        scores   = pred_raw[pat]
        calls    = hi.convert_pred(list(scores.values()))
        df_pat   = pd.DataFrame({
            'patient_id':        pat,
            'HLA':               [a.replace('HLA-','') for a in scores.keys()],
            'score':             list(scores.values()),
            'predicted_carrier': calls.astype(bool)
        })
        df_pat.to_csv(out_file, index=False)
        results.append(df_pat)
    except Exception as e:
        print(f'ERROR {pat}: {e}')

    elapsed = time.time() - t0
    eta     = elapsed / (i+1) * (len(all_patients) - i - 1)
    if (i+1) % 20 == 0:
        print(f'{i+1}/{len(all_patients)} | {elapsed/60:.1f} min | ETA {eta/60:.1f} min')

thnet_all = pd.concat(results, ignore_index=True)
thnet_all.to_csv(RESULTS_DIR + 'thnet_rosati_predictions.csv', index=False)

print(f'\nDone in {(time.time()-t0)/60:.1f} min')
print(f'Total predictions: {len(thnet_all):,} | Patients: {thnet_all["patient_id"].nunique()}')

## 5. Results overview

Excluding DPA1*01:03, the score distribution confirms good model calibration: predicted carriers have mean score 0.864 (min 0.130) while non-carriers have median score 0.001. Maximum predicted alleles per patient is 14, consistent with diploid biology (2 alleles × 7 loci).

In [ ]:
thnet = pd.read_csv(RESULTS_DIR + 'thnet_rosati_predictions.csv')
thnet['group'] = thnet['patient_id'].apply(get_group)
thnet_clean    = thnet[thnet['HLA'] != 'DPA1*01:03'].copy()

print('=== SCORE DISTRIBUTION ===')
for label, mask in [('Carriers',     thnet_clean['predicted_carrier']==True),
                    ('Non-carriers',  thnet_clean['predicted_carrier']==False)]:
    print(f'{label}:')
    print(thnet_clean[mask]['score'].describe().round(3))
    print()

print('=== ALLELES PER PATIENT ===')
apper = thnet_clean[thnet_clean['predicted_carrier']==True].groupby('patient_id').size()
print(apper.describe().round(1))
print(f'Patients with >14 alleles (biologically implausible): {(apper>14).sum()}')

print('\n=== COVERAGE BY GROUP ===')
for grp in ['Healthy','CD','UC','CRC']:
    sub  = thnet_clean[thnet_clean['group']==grp]
    pats = sub['patient_id'].nunique()
    w_sig = sub[sub['predicted_carrier']==True]['patient_id'].nunique()
    mean_a = sub[sub['predicted_carrier']==True].groupby('patient_id').size().mean()
    print(f'  {grp:<10}: {pats} patients | {w_sig} with predictions | mean {mean_a:.1f} alleles')

print('\n=== TOP 20 ALLELES BY PREDICTED FREQUENCY ===')
top = thnet_clean[thnet_clean['predicted_carrier']==True]\
      .groupby('HLA').agg(n=('patient_id','count'), mean_score=('score','mean'))\
      .sort_values('n', ascending=False).head(20)
top['freq%'] = (top['n'] / thnet_clean['patient_id'].nunique() * 100).round(1)
top['mean_score'] = top['mean_score'].round(3)
print(top.to_string())

## 6. HLAGuessr vs THNet — method comparison

Comparison restricted to 94 shared alleles (excluding DPA1*01:03). Agreement = both methods predict `carrier = True` for the same patient-allele pair. Mean Jaccard similarity ranges from 0.357 (UC) to 0.526 (Healthy), indicating moderate but meaningful concordance — though note that HLAGuessr's predictions here are not fully independent of this cohort, since Rosati's data was used in HLAGuessr's own training (see notebook 0 caveat). THNet's training set is fully external to this cohort.

In [ ]:
hlag  = pd.read_csv(HLAG_PREDS)
hlag['group'] = hlag['patient_id'].apply(get_group)

shared_alleles = sorted(
    set(thnet_clean['HLA'].unique()) &
    set(hlag['HLA'].unique()) - {'DPA1*01:03'}
)
print(f'Shared alleles for comparison: {len(shared_alleles)}')

thnet_by_pat = thnet_clean[
    thnet_clean['HLA'].isin(shared_alleles) & (thnet_clean['predicted_carrier']==True)
].groupby('patient_id')['HLA'].apply(set)

hlag_by_pat = hlag[
    hlag['HLA'].isin(shared_alleles) & (hlag['predicted_carrier']==True)
].groupby('patient_id')['HLA'].apply(set)

comp_rows = []
for pat in sorted(set(thnet_clean['patient_id']) & set(hlag['patient_id'])):
    h = hlag_by_pat.get(pat, set())
    t = thnet_by_pat.get(pat, set())
    comp_rows.append({
        'patient_id': pat, 'group': get_group(pat),
        'n_hlag': len(h), 'n_thnet': len(t),
        'n_agree': len(h & t),
        'jaccard': len(h & t) / len(h | t) if len(h | t) > 0 else 0
    })
comp = pd.DataFrame(comp_rows)

print('\n=== AGREEMENT BY GROUP ===')
for grp in ['Healthy','CD','UC','CRC']:
    sub = comp[comp['group']==grp]
    print(f'  {grp} (n={len(sub)}): '
          f'HLAGuessr {sub["n_hlag"].mean():.1f} | '
          f'THNet {sub["n_thnet"].mean():.1f} | '
          f'Agreement {sub["n_agree"].mean():.1f} | '
          f'Jaccard {sub["jaccard"].mean():.3f}')

## 7. High-confidence genotype — both methods ≥90%

Restricting to alleles where both methods predict `carrier = True` with scores ≥90% yields the most reliable predicted genotypes: **192/267 patients (71.9%)** have at least one confirmed allele. Coverage by group: Healthy 80.8%, CD 71.9%, UC 70.0%, CRC 13.3%.

In [ ]:
thnet_90 = thnet_clean[
    thnet_clean['HLA'].isin(shared_alleles) &
    (thnet_clean['predicted_carrier']==True) &
    (thnet_clean['score'] >= 0.90)
][['patient_id','HLA','score']].rename(columns={'score':'THNet_score'})

hlag_90 = hlag[
    hlag['HLA'].isin(shared_alleles) &
    (hlag['predicted_carrier']==True) &
    (hlag['probability'] >= 0.90)
][['patient_id','HLA','probability']].rename(columns={'probability':'HLAGuessr_prob'})

agree_90 = pd.merge(hlag_90, thnet_90, on=['patient_id','HLA'], how='inner')
agree_90['group']       = agree_90['patient_id'].apply(get_group)
agree_90['HLAGuessr_%'] = (agree_90['HLAGuessr_prob'] * 100).round(1)
agree_90['THNet_%']     = (agree_90['THNet_score']    * 100).round(1)
agree_90['Delta']       = (agree_90['HLAGuessr_%'] - agree_90['THNet_%']).abs().round(1)

print(f'Agreements at >=90%: {len(agree_90):,}')
print(f'Patients covered:    {agree_90["patient_id"].nunique()}/267 '
      f'({agree_90["patient_id"].nunique()/267*100:.1f}%)')
print(f'Mean Delta:          {agree_90["Delta"].mean():.1f}%')
print()
n_tot = {'Healthy':99,'CD':114,'UC':40,'CRC':15}
for grp in ['Healthy','CD','UC','CRC']:
    n = agree_90[agree_90['group']==grp]['patient_id'].nunique()
    print(f'  {grp:<10}: {n}/{n_tot[grp]} ({n/n_tot[grp]*100:.1f}%)')
print()
print('Top 15 alleles confirmed by both methods at >=90%:')
print(agree_90['HLA'].value_counts().head(15).to_string())

## 8. Conclusions

- **THNet inferred HLA genotypes for 267/267 Rosati patients** using TCRβ only, covering 208 alleles across 7 loci.

- **DPA1*01:03 is a model artefact** — predicted True in 100% of patients, excluded from all analyses. Reflects class imbalance in THNet training (~95% DPA1*01:03 carrier frequency in Europeans).

- **Method agreement (HLAGuessr vs THNet)** on 94 shared alleles: Jaccard 0.357–0.526 by group. Note this is not a comparison between two equally independent methods — HLAGuessr's training included this cohort, while THNet's did not (see notebook 0 caveat).

- **High-confidence genotypes** (both methods ≥90%): 192/267 patients (71.9%), maximum 14 alleles per patient — biologically consistent with diploid HLA.

- **CRC group**: near-universal failure (median 8 clones/patient) — insufficient T-cell data for reliable inference.

- **Ground-truth HLA genotypes for the CD and Healthy groups (192 patients) were subsequently obtained directly from Elisa Rosati** and used to validate THNet's predictions independently — since THNet had no prior exposure to this cohort, this validation is genuinely blind. See `rosati_qc_hla_validation.ipynb` for the full validation and `README.md` for headline accuracy results (THNet: 47.1% sensitivity at 98.5% precision).